# Shor's Algorithm and the Quantum Threat to Classical Cryptography

In 1994, mathematician Peter Shor published an algorithm that would reshape the foundations of modern cryptography.
His algorithm demonstrated that a sufficiently large quantum computer could factor integers in polynomial time,
reducing a problem that takes classical computers billions of years to one solvable in hours or minutes.
This is not a theoretical curiosity: the entire security of RSA encryption rests on the assumption that
factoring the product of two large primes is computationally infeasible. Elliptic Curve Cryptography (ECC)
faces a parallel threat through the quantum discrete logarithm problem.

The urgency is compounded by the **Harvest Now, Decrypt Later (HNDL)** threat model. Nation-state
adversaries and sophisticated attackers are already intercepting and storing encrypted traffic today,
betting that future quantum computers will let them decrypt it retroactively. For data with long
retention requirements (healthcare records, government secrets, financial instruments, intellectual property),
this means the threat window is **not** when a cryptographically relevant quantum computer (CRQC) arrives;
it is **today**, the moment the data is transmitted.

The NSA's CNSA 2.0 advisory mandates that all new National Security Systems use post-quantum algorithms
by **2027**, with full transition by 2033. NIST deprecated RSA and ECC for new applications in 2024
and will disallow them entirely after 2035. The migration window is closing.

This notebook walks through Shor's algorithm step by step, visualizes the qubit gap between today's
hardware and what is needed to break RSA, and demonstrates Zipminator's HNDL risk calculator to
quantify the exposure of real-world data assets.

## The Mathematics of Shor's Algorithm

Shor's algorithm converts the factoring problem into a **period-finding** problem, which quantum
computers solve exponentially faster than classical ones. The procedure has four key stages:

**1. Random base selection.** Choose a random integer $a$ where $1 < a < N$ and $\gcd(a, N) = 1$.
If $\gcd(a, N) \neq 1$, we already found a factor and can stop.

**2. Quantum period finding.** The core quantum subroutine finds the period $r$ of the function
$f(x) = a^x \bmod N$. This is the smallest positive integer $r$ such that $a^r \equiv 1 \pmod{N}$.
The quantum circuit prepares a superposition of all $x$ values, computes $a^x \bmod N$ in a second
register via **modular exponentiation**, then applies the **inverse Quantum Fourier Transform (QFT)**
to the first register. Measurement yields a value $y \approx k \cdot 2^n / r$ for some integer $k$.

**3. Classical post-processing.** Use the **continued fractions algorithm** to extract $r$ from the
measured $y/2^n$. The convergents of the continued fraction expansion give candidate periods.

**4. Factor extraction.** If $r$ is even, compute $\gcd(a^{r/2} \pm 1, N)$. With probability
$\geq 1/2$, at least one of these is a non-trivial factor of $N$.

The computational complexity is $O((\log N)^2 (\log \log N)(\log \log \log N))$ for the quantum
part, compared to the best known classical algorithm (General Number Field Sieve) at
$O\left(\exp\left(\left(\frac{64}{9}\right)^{1/3} (\ln N)^{1/3} (\ln \ln N)^{2/3}\right)\right)$.
For RSA-2048, this is the difference between hours on a quantum computer and trillions of years classically.

## Environment Setup

This notebook uses Qiskit for quantum circuit construction and simulation. If Qiskit is not
installed, the circuit-building cells will not execute, but the analysis and visualization cells
use only Plotly and NumPy and will run independently.

Install the quantum extras with: `uv pip install zipminator[quantum]`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, "..")
from _helpers.viz import *

import numpy as np

QISKIT_AVAILABLE = False
try:
    import qiskit
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import QFT
    QISKIT_AVAILABLE = True
    print(f"Qiskit version: {qiskit.__version__}")
except ImportError:
    print("Qiskit not installed. Circuit cells will be skipped.")

print(f"NumPy: {np.__version__}")
print(f"Plotly: {import_module('plotly').__version__}" if False else "Plotly: loaded via viz helper")
print("Quantum dark theme: active")

## Factoring N = 15 with Shor's Algorithm

N = 15 is the canonical demonstration target for Shor's algorithm. It is the smallest composite
number that is a product of two distinct odd primes (3 and 5). We use base $a = 7$. The sequence
$7^x \bmod 15$ produces: 1, 7, 4, 13, 1, 7, 4, 13, ... giving a period $r = 4$.

From this: $\gcd(7^{4/2} + 1, 15) = \gcd(50, 15) = 5$ and $\gcd(7^{4/2} - 1, 15) = \gcd(48, 15) = 3$.

The circuit uses 8 counting qubits and 4 work qubits, totaling 12 qubits.

In [ ]:
# Visualize the modular exponentiation sequence: 7^x mod 15
x_vals = list(range(16))
y_vals = [pow(7, x, 15) for x in x_vals]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_vals, y=y_vals, mode="lines+markers",
    line=dict(color=ZM_COLORS["cyan"], width=3),
    marker=dict(size=10, color=ZM_COLORS["cyan"],
                line=dict(color="white", width=1)),
    name="7^x mod 15",
    hovertemplate="x=%{x}<br>7^x mod 15 = %{y}<extra></extra>",
))

# Highlight the period
for i in range(0, 16, 4):
    fig.add_vrect(x0=i-0.3, x1=i+0.3, fillcolor=ZM_COLORS["violet"],
                  opacity=0.15, line_width=0)

fig.update_layout(
    title="Modular Exponentiation: 7<sup>x</sup> mod 15 (Period r = 4)",
    xaxis_title="Exponent x",
    yaxis_title="7<sup>x</sup> mod 15",
    height=400,
    annotations=[dict(x=2, y=16, text="Period r = 4", showarrow=False,
                      font=dict(color=ZM_COLORS["violet"], size=14))]
)
fig.show()

## Qubit Requirements: The Chasm Between Toy Examples and Real RSA

Factoring N = 15 on 12 qubits is compelling, but it is separated from breaking real cryptography
by an enormous gap. The number of **logical** qubits scales as roughly $2n + O(\log n)$. For RSA-2048,
that means approximately 4,098 logical qubits.

However, each logical qubit must be encoded in many **physical** qubits using quantum error correction.
Current estimates require 1,000 to 10,000 physical qubits per logical qubit. This puts RSA-2048
at approximately 4 to 20 million physical qubits.

IBM's current largest processor has ~1,100 qubits (Condor, 2023). Their roadmap targets
100,000 qubits by 2033, still 1-2 orders of magnitude short.

In [ ]:
targets = ['N=15 (4 bits)', 'N=21 (5 bits)', 'RSA-1024', 'RSA-2048', 'RSA-4096']
logical_qubits  = [12, 15, 2_050, 4_098, 8_196]
physical_qubits = [12, 15, 4_000_000, 20_000_000, 100_000_000]
bar_colors = [ZM_COLORS["emerald"], ZM_COLORS["emerald"],
              ZM_COLORS["amber"], ZM_COLORS["rose"], ZM_COLORS["rose"]]

fig = zm_subplots(1, 2, titles=["Logical Qubits Required", "Physical Qubits (with QEC)"])

# Logical qubits
fig.add_trace(go.Bar(
    y=targets, x=logical_qubits, orientation="h",
    marker_color=bar_colors, name="Logical",
    text=[f"{v:,}" for v in logical_qubits], textposition="outside",
    hovertemplate="%{y}: %{x:,} logical qubits<extra></extra>",
), row=1, col=1)

# Physical qubits
fig.add_trace(go.Bar(
    y=targets, x=physical_qubits, orientation="h",
    marker_color=bar_colors, name="Physical",
    text=[f"{v:,}" if v < 1e6 else f"{v/1e6:.0f}M" for v in physical_qubits],
    textposition="outside",
    hovertemplate="%{y}: %{x:,} physical qubits<extra></extra>",
), row=1, col=2)

# Hardware milestone lines
fig.add_vline(x=156, line_dash="dash", line_color=ZM_COLORS["cyan"],
              annotation_text="IBM 156q (2025)", row=1, col=1)
fig.add_vline(x=1100, line_dash="dash", line_color=ZM_COLORS["violet"],
              annotation_text="IBM Condor 1,100q", row=1, col=1)
fig.add_vline(x=100_000, line_dash="dash", line_color=ZM_COLORS["amber"],
              annotation_text="IBM 100K target (2033)", row=1, col=2)

fig.update_xaxes(type="log", row=1, col=1)
fig.update_xaxes(type="log", row=1, col=2)
fig.update_layout(
    title="The Quantum Gap: Current Hardware vs RSA Requirements",
    height=500, showlegend=False,
)
fig.show()

## The HNDL Timeline: When Does Encrypted Data Become Vulnerable?

The Harvest Now, Decrypt Later threat does not begin when a CRQC is built. It begins the moment
sensitive data is transmitted over a channel that an adversary can intercept:

$$\text{At risk if: } T_{\text{harvest}} + T_{\text{retention}} > T_{\text{CRQC}}$$

The timeline below maps the critical milestones from NIST standardization through RSA deprecation.

In [ ]:
milestones = [
    (2024, "NIST PQC Standards Finalized", ZM_COLORS["emerald"], 1),
    (2025, "Current QC: ~1,100 qubits", ZM_COLORS["cyan"], -1),
    (2027, "CNSA 2.0 Deadline", ZM_COLORS["amber"], 1),
    (2030, "NIST Deprecates RSA & ECC", ZM_COLORS["amber"], -1),
    (2033, "IBM 100K Qubit Target", ZM_COLORS["violet"], 1),
    (2035, "NIST Disallows RSA & ECC", ZM_COLORS["rose"], -1),
]

fig = go.Figure()

# Timeline axis
fig.add_trace(go.Scatter(
    x=[2023, 2036], y=[0, 0], mode="lines",
    line=dict(color="#94a3b8", width=3), showlegend=False,
))

# HNDL danger zone
fig.add_vrect(x0=2024, x1=2035, fillcolor=ZM_COLORS["rose"], opacity=0.06)
fig.add_annotation(x=2029.5, y=-0.85, text="HNDL DANGER ZONE",
                   font=dict(color=ZM_COLORS["rose"], size=14), showarrow=False)

for year, label, color, direction in milestones:
    # Connector line
    fig.add_trace(go.Scatter(
        x=[year, year], y=[0, direction * 0.4], mode="lines",
        line=dict(color=color, width=2), showlegend=False,
    ))
    # Marker
    fig.add_trace(go.Scatter(
        x=[year], y=[0], mode="markers",
        marker=dict(size=14, color=color, line=dict(color="white", width=2)),
        showlegend=False, hovertext=f"{year}: {label}", hoverinfo="text",
    ))
    # Label
    fig.add_annotation(
        x=year, y=direction * 0.55, text=f"<b>{year}</b><br>{label}",
        font=dict(color=color, size=11), showarrow=False,
    )

# NOW marker
fig.add_vline(x=2026, line_dash="dot", line_color=ZM_COLORS["cyan"], opacity=0.5)
fig.add_annotation(x=2026, y=0.9, text="<b>NOW</b>",
                   font=dict(color=ZM_COLORS["cyan"], size=13), showarrow=False)

fig.update_layout(
    title="The Post-Quantum Migration Timeline",
    xaxis=dict(range=[2023, 2036.5], dtick=1),
    yaxis=dict(range=[-1.1, 1.1], visible=False),
    height=450, showlegend=False,
    template=ZM_TEMPLATE,
)
fig.show()

## HNDL Risk Calculator

Zipminator includes an HNDL risk scoring engine that quantifies exposure based on four factors:
data sensitivity, retention period, current encryption, and industry multiplier.

The risk score ranges from 0 to 100: LOW (< 25), MEDIUM (25-50), HIGH (50-75), CRITICAL (> 75).

In [ ]:
from zipminator.hndl_risk import HNDLCalculator

calc = HNDLCalculator()

scenarios = [
    {'name': 'Healthcare (HIPAA)',   'sensitivity': 'top_secret', 'years': 50, 'industry': 'healthcare'},
    {'name': 'Financial (SOX)',      'sensitivity': 'confidential', 'years': 30, 'industry': 'finance'},
    {'name': 'Government (Secret)',  'sensitivity': 'top_secret', 'years': 75, 'industry': 'government'},
]

names, classical_risks, pqc_risks = [], [], []
for s in scenarios:
    names.append(s['name'])
    r_classical = calc.calculate(data_sensitivity=s['sensitivity'], retention_years=s['years'],
                                 current_encryption='aes256', industry=s['industry'])
    r_pqc = calc.calculate(data_sensitivity=s['sensitivity'], retention_years=s['years'],
                            current_encryption='kyber768', industry=s['industry'])
    classical_risks.append(r_classical.overall_risk)
    pqc_risks.append(r_pqc.overall_risk)

fig = go.Figure()
fig.add_trace(go.Bar(name="AES-256 (Classical)", x=names, y=classical_risks,
                     marker_color=ZM_COLORS["rose"], text=[f"{v:.0f}%" for v in classical_risks],
                     textposition="outside"))
fig.add_trace(go.Bar(name="ML-KEM-768 (PQC)", x=names, y=pqc_risks,
                     marker_color=ZM_COLORS["emerald"], text=[f"{v:.0f}%" for v in pqc_risks],
                     textposition="outside"))

# Risk level threshold lines
fig.add_hline(y=75, line_dash="dash", line_color=ZM_COLORS["rose"], opacity=0.5,
              annotation_text="CRITICAL", annotation_position="right")
fig.add_hline(y=50, line_dash="dash", line_color=ZM_COLORS["amber"], opacity=0.5,
              annotation_text="HIGH", annotation_position="right")
fig.add_hline(y=25, line_dash="dash", line_color=ZM_COLORS["cyan"], opacity=0.3,
              annotation_text="MEDIUM", annotation_position="right")

fig.update_layout(
    title="HNDL Risk Score: Classical vs Post-Quantum Encryption",
    yaxis_title="Risk Score (%)", yaxis_range=[0, 105],
    barmode="group", height=500,
    template=ZM_TEMPLATE,
)
fig.show()

## Key Size Comparison: Classical vs Post-Quantum Algorithms

ML-KEM-768 has a public key of 1,184 bytes vs 256 bytes for RSA-2048 (4.6x increase).
For context, a single HD video frame is ~300KB. The trade-off is negligible in practice.

In [ ]:
algorithms = [
    ('RSA-2048',     256,    256,    'KEM',  False),
    ('RSA-4096',     512,    512,    'KEM',  False),
    ('ECC P-256',     64,     64,    'KEM',  False),
    ('ML-KEM-768',  1184,   1088,   'KEM',  True),
    ('ML-KEM-1024', 1568,   1568,   'KEM',  True),
    ('ECDSA P-256',   64,     64,   'Sig',  False),
    ('ML-DSA-65',   1952,   3293,   'Sig',  True),
    ('SLH-DSA-128f', 32,   17088,  'Sig',  True),
]

names = [a[0] for a in algorithms]
pk_sizes = [a[1] for a in algorithms]
ct_sizes = [a[2] for a in algorithms]
q_safe = [a[4] for a in algorithms]
colors_pk = [ZM_COLORS["cyan"] if qs else ZM_COLORS["rose"] for qs in q_safe]
colors_ct = [ZM_COLORS["violet"] if qs else ZM_COLORS["amber"] for qs in q_safe]

fig = zm_subplots(2, 1, titles=["Public Key Size (bytes)", "Ciphertext / Signature Size (bytes)"],
                  shared_xaxes=True, vertical_spacing=0.12)

fig.add_trace(go.Bar(
    x=names, y=pk_sizes, marker_color=colors_pk, name="Public Key",
    text=[f"{v:,}B" for v in pk_sizes], textposition="outside",
    hovertemplate="%{x}: %{y:,} bytes<extra>Public Key</extra>",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=names, y=ct_sizes, marker_color=colors_ct, name="CT / Sig",
    text=[f"{v:,}B" if v < 10000 else f"{v/1000:.1f}KB" for v in ct_sizes],
    textposition="outside",
    hovertemplate="%{x}: %{y:,} bytes<extra>Ciphertext/Signature</extra>",
), row=2, col=1)

fig.update_yaxes(type="log", row=1, col=1)
fig.update_yaxes(type="log", row=2, col=1)
fig.update_layout(
    title="Algorithm Comparison: Key & Ciphertext Sizes",
    height=700, showlegend=False,
)
fig.show()

## Interactive: Algorithm Security Over Time

Watch how the security of classical algorithms degrades as quantum computing advances,
while post-quantum algorithms remain secure.

In [ ]:
# Animated chart: Security level over time as quantum computing scales
years = list(range(2024, 2041))
algorithms_timeline = {
    "RSA-2048": [128]*3 + [100, 80, 60, 40, 30, 20, 10, 5, 2, 1, 0, 0, 0, 0],
    "ECC P-256": [128]*3 + [90, 70, 50, 30, 20, 10, 5, 2, 1, 0, 0, 0, 0, 0],
    "AES-256": [256]*6 + [200, 180, 170, 160, 150, 140, 135, 130, 128, 128, 128],
    "ML-KEM-768": [192]*len(years),
    "ML-KEM-1024": [256]*len(years),
}

frames_data = []
for i, year in enumerate(years):
    algo_names = list(algorithms_timeline.keys())
    values = [algorithms_timeline[a][i] for a in algo_names]
    colors = [ZM_COLORS["emerald"] if v > 100 else
              ZM_COLORS["amber"] if v > 50 else
              ZM_COLORS["rose"] for v in values]
    frames_data.append({"name": str(year), "x": algo_names, "y": values, "color": colors})

fig = zm_animated_bar(frames_data, title="Security Level (bits) as Quantum Computing Scales",
                      x_label="Algorithm", y_label="Effective Security (bits)")
fig.update_layout(yaxis_range=[0, 280], height=500)
fig.show()

## Conclusion: The Time to Act Is Now

This notebook demonstrated that Shor's algorithm is not science fiction. The mathematics are proven,
the circuits are well-understood, and the only barrier is hardware scale.

Zipminator implements **ML-KEM-768 (FIPS 203)**, the NIST-standardized post-quantum key encapsulation
mechanism. It is not a future product; it is available today.

```
pip install zipminator          # Core PQC encryption
pip install zipminator[quantum] # Quantum extras (Qiskit, entropy harvesting)
```